In [0]:
%pip install "databricks-openai[memory]>=0.13.0" "databricks-sdk>=0.70.0"
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
import json

w = WorkspaceClient()

PROJECT_ID = "<your lakebase database project ID/name>"

# List branches to find the correct branch ID
branches = w.api_client.do("GET", f"/api/2.0/postgres/projects/{PROJECT_ID}/branches")
print("Branches:")
for b in branches.get("branches", []):
    print(f"  {b['name']}")
    branch_id = b["name"].split("/branches/")[-1]

    # List endpoints for each branch
    endpoints = w.api_client.do("GET", f"/api/2.0/postgres/projects/{PROJECT_ID}/branches/{branch_id}/endpoints")
    for ep in endpoints.get("endpoints", []):
        print(f"    Endpoint: {ep['name']}")
        print(f"    Host: {ep.get('status', {}).get('host', 'N/A')}")

In [ ]:
# DO NOT RUN — the app creates agent memory tables automatically on first startup.

# from databricks_openai.agents.session import AsyncDatabricksSession

# # Replace with YOUR Lakebase endpoint path from Step 2
# LAKEBASE_ENDPOINT = "projects/<project name>/branches/<branch name>/endpoints/primary"

# async def create_agent_tables():
#     session = AsyncDatabricksSession(
#         session_id="__setup__",
#         autoscaling_endpoint=LAKEBASE_ENDPOINT,
#         schema="agent_openai_memory",
#     )
#     await session._ensure_tables()
#     print("✓ Backend agent memory tables created (schema: agent_openai_memory)")

# await create_agent_tables()

In [0]:
from databricks.sdk import WorkspaceClient
import json

w = WorkspaceClient()

# List databases for the project/branch
response = w.api_client.do(
    "GET",
    "/api/2.0/postgres/projects/<project name>/branches/production/databases"
)
print(json.dumps(response, indent=2))

In [ ]:
# DO NOT RUN — the app creates chat history tables (ai_chatbot schema) automatically via Drizzle on first startup.

# from databricks_ai_bridge.lakebase import LakebaseClient

# client = LakebaseClient(
#     autoscaling_endpoint="projects/<project name>/branches/<branch name>/endpoints/primary"
# )

# client.execute("CREATE SCHEMA IF NOT EXISTS ai_chatbot;")

# client.execute('''
# CREATE TABLE IF NOT EXISTS ai_chatbot."User" (
#     id text PRIMARY KEY,
#     email varchar(64) NOT NULL
# );
# ''')
# client.execute('''
# CREATE TABLE IF NOT EXISTS ai_chatbot."Chat" (
#     id uuid PRIMARY KEY DEFAULT gen_random_uuid(),
#     "createdAt" timestamp NOT NULL,
#     title text NOT NULL,
#     "userId" text NOT NULL,
#     visibility varchar DEFAULT 'private' NOT NULL,
#     "lastContext" jsonb
# );
# ''')
# client.execute('''
# CREATE TABLE IF NOT EXISTS ai_chatbot."Message" (
#     id uuid PRIMARY KEY DEFAULT gen_random_uuid(),
#     "chatId" uuid NOT NULL REFERENCES ai_chatbot."Chat"(id),
#     role varchar NOT NULL,
#     parts json NOT NULL,
#     attachments json NOT NULL,
#     "createdAt" timestamp NOT NULL,
#     "traceId" text
# );
# ''')
# client.execute('''
# CREATE TABLE IF NOT EXISTS ai_chatbot."Vote" (
#     "chatId" uuid NOT NULL REFERENCES ai_chatbot."Chat"(id),
#     "messageId" uuid NOT NULL REFERENCES ai_chatbot."Message"(id),
#     "isUpvoted" boolean NOT NULL,
#     PRIMARY KEY("chatId", "messageId")
# );
# ''')

# print("✓ Schema ai_chatbot and all tables created.")
# client.close()

In [ ]:
# DO NOT RUN — permissions are not needed; the app owns its own tables.

# from databricks_ai_bridge.lakebase import LakebaseClient

# client = LakebaseClient(
#     autoscaling_endpoint="projects/<project name>/branches/<branch name>/endpoints/primary"
# )

# grants = [
#     # Backend agent memory schema
#     "GRANT USAGE ON SCHEMA agent_openai_memory TO PUBLIC;",
#     "GRANT ALL ON ALL TABLES IN SCHEMA agent_openai_memory TO PUBLIC;",
#     "GRANT USAGE, SELECT, UPDATE ON ALL SEQUENCES IN SCHEMA agent_openai_memory TO PUBLIC;",
#     "ALTER DEFAULT PRIVILEGES IN SCHEMA agent_openai_memory GRANT ALL ON TABLES TO PUBLIC;",
#     "ALTER DEFAULT PRIVILEGES IN SCHEMA agent_openai_memory GRANT USAGE, SELECT, UPDATE ON SEQUENCES TO PUBLIC;",
#     # Frontend chat history schema
#     "GRANT USAGE ON SCHEMA ai_chatbot TO PUBLIC;",
#     "GRANT ALL ON ALL TABLES IN SCHEMA ai_chatbot TO PUBLIC;",
#     "GRANT USAGE, SELECT, UPDATE ON ALL SEQUENCES IN SCHEMA ai_chatbot TO PUBLIC;",
#     "ALTER DEFAULT PRIVILEGES IN SCHEMA ai_chatbot GRANT ALL ON TABLES TO PUBLIC;",
#     "ALTER DEFAULT PRIVILEGES IN SCHEMA ai_chatbot GRANT USAGE, SELECT, UPDATE ON SEQUENCES TO PUBLIC;",
# ]

# for grant in grants:
#     try:
#         client.execute(grant)
#     except Exception as e:
#         print(f"Warning on '{grant[:50]}...': {e}")

# print("✓ Permissions granted to PUBLIC for both schemas")
# client.close()